In [9]:
from langchain_core.documents import Document



In [10]:
from langchain_community.document_loaders import PyMuPDFLoader
file_path = "data/NepalSambidhan.pdf"
loader = PyMuPDFLoader(file_path)
docs = loader.load()
docs


[Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2024-09-18T12:48:20+05:45', 'source': 'data/NepalSambidhan.pdf', 'file_path': 'data/NepalSambidhan.pdf', 'total_pages': 183, 'format': 'PDF 1.5', 'title': 'kmf}Hbf/L s;"/df ;hfo lgwf{/0f tyf sfof{Gjog ug]{ ;DaGwdf Joj:yf ug{ ag]sf] ljw]os', 'author': 'Ravi Sharma Aryal', 'subject': '', 'keywords': '', 'moddate': '2024-09-18T12:48:20+05:45', 'trapped': '', 'modDate': "D:20240918124820+05'45'", 'creationDate': "D:20240918124820+05'45'", 'page': 0}, page_content='www.lawcommission.gov.np \n1 \n \nनेपालको संविधान \n \nनेपाल राजपत्रमा प्रकाशन मममि  \n२०७२।०६।०३  \nसंशोधन   \n \n \n \n \n \n \n  प्रमाणीकरण र प्रकाशन मममि  \n१. नेपालको संविधान (पविलो संशोधन), २०७२  \n \n \n  २०७२।११।१६ \n२. नेपालको संविधान (दोस्रो संशोधन), २०७७  \n \n   \n  २०७७।०३।०४ \n  \n \nप्रस्िािना \nिामी सािवभौमसत्तासम्पन् न नेपाली जनिा;  \nनेपालको स्ििन्त्रिा, सािवभौममकिा, भौगोमलक अखण्डिा, राविय एकिा, स्िाधीनिा 

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 200,
        separators = ["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    return split_docs


In [14]:
chunks = split_documents(docs)
chunks
len(chunks)

352

In [1]:
from dotenv import load_dotenv
import os
from google import genai
import numpy as np
load_dotenv()
gemini_api_key=os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=gemini_api_key)
client

In [5]:
class EmbeddingManager():

    def __init__(self, model_name: str = "gemini-embedding-001"):
        self.model_name = model_name
        self.client = None
        self._load_client()

    def _load_client(self):
        try:
            api_key = os.getenv("GEMINI_API_KEY")
            if not api_key:
                raise ValueError("API_KEY not found in environment variables.")
            print(f"Initializing Google Gemini client for model: {self.model_name}")
            self.client = genai.Client(api_key=api_key)
            print("Client initialized successfully.")
        except Exception as e:
            print(f"Error initializing Gemini client: {e}")
            raise

    def generate_embeddings(self, chunks: list[str]) -> np.ndarray:
        """Generate embeddings for a list of text chunks."""
        if not self.client:
            raise ValueError("Client not initialized.")
        try:
            response = self.client.models.embed_content(
            model=self.model_name,
            contents=chunks
            )
        except Exception as e:
            print(f"Embedding Failed due to {e}")

        # Extract the embedding vectors from the response and convert to ndarray
        vectors = [embedding.values for embedding in response.embeddings]
        return np.array(vectors)


In [6]:
embedding = EmbeddingManager()
embedding

Initializing Google Gemini client for model: gemini-embedding-001
Client initialized successfully.
